In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# AI Compute & Margin Analysis\n",
    "\n",
    "This notebook reproduces the estimates used in the two-slide deck:\n",
    "1. OpenAI / Anthropic GPU park estimate and training vs inference split\n",
    "2. Atlas Cloud margin estimate on DeepSeek V4 Pro\n",
    "\n",
    "All numbers are scenario-based and can be adjusted via assumptions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "plt.style.use('seaborn-v0_8-whitegrid')\n",
    "pd.set_option('display.max_columns', 50)\n",
    "pd.set_option('display.width', 120)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1) OpenAI / Anthropic GPU park estimate\n",
    "\n",
    "Assumptions are broad, because public information is incomplete.\n",
    "We estimate total GPU-equivalents and then split into training vs inference."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "companies = {\n",
    "    'OpenAI': {\n",
    "        'total_low': 900_000,\n",
    "        'total_base': 1_100_000,\n",
    "        'total_high': 1_400_000,\n",
    "        'training_share_low': 0.40,\n",
    "        'training_share_base': 0.48,\n",
    "        'training_share_high': 0.55,\n",
    "    },\n",
    "    'Anthropic': {\n",
    "        'total_low': 250_000,\n",
    "        'total_base': 350_000,\n",
    "        'total_high': 450_000,\n",
    "        'training_share_low': 0.25,\n",
    "        'training_share_base': 0.32,\n",
    "        'training_share_high': 0.40,\n",
    "    }\n",
    "}\n",
    "\n",
    "rows = []\n",
    "for company, a in companies.items():\n",
    "    for scenario in ['low', 'base', 'high']:\n",
    "        total = a[f'total_{scenario}']\n",
    "        train_share = a[f'training_share_{scenario}']\n",
    "        rows.append({\n",
    "            'company': company,\n",
    "            'scenario': scenario,\n",
    "            'total_gpu_eq': total,\n",
    "            'training_gpu_eq': total * train_share,\n",
    "            'inference_gpu_eq': total * (1 - train_share),\n",
    "            'training_share': train_share,\n",
    "            'inference_share': 1 - train_share,\n",
    "        })\n",
    "\n",
    "gpu_df = pd.DataFrame(rows)\n",
    "gpu_df"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "pivot = gpu_df.pivot(index='company', columns='scenario', values=['total_gpu_eq', 'training_gpu_eq', 'inference_gpu_eq'])\n",
    "pivot"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Helper: formatted summary table"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def fmt_m(x):\n",
    "    return f'{x/1e6:.2f}M'\n",
    "\n",
    "summary = []\n",
    "for company, a in companies.items():\n",
    "    summary.append({\n",
    "        'company': company,\n",
    "        'total_low': fmt_m(a['total_low']),\n",
    "        'total_base': fmt_m(a['total_base']),\n",
    "        'total_high': fmt_m(a['total_high']),\n",
    "        'training_base': fmt_m(a['total_base'] * a['training_share_base']),\n",
    "        'inference_base': fmt_m(a['total_base'] * (1 - a['training_share_base'])),\n",
    "    })\n",
    "\n",
    "pd.DataFrame(summary)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2) Atlas Cloud margin analysis on DeepSeek V4 Pro\n",
    "\n",
    "Public price:\n",
    "- Input: $1.68 / 1M tokens\n",
    "- Output: $3.38 / 1M tokens\n",
    "\n",
    "We estimate cost-to-serve ranges and compute gross margin."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "price_input = 1.68\n",
    "price_output = 3.38\n",
    "\n",
    "# Assumed cost-to-serve range per 1M tokens\n",
    "cost_input_low, cost_input_base, cost_input_high = 0.30, 0.50, 0.80\n",
    "cost_output_low, cost_output_base, cost_output_high = 0.90, 1.50, 2.20\n",
    "\n",
    "margin_input = {\n",
    "    'low': (price_input - cost_input_high) / price_input,\n",
    "    'base': (price_input - cost_input_base) / price_input,\n",
    "    'high': (price_input - cost_input_low) / price_input,\n",
    "}\n",
    "\n",
    "margin_output = {\n",
    "    'low': (price_output - cost_output_high) / price_output,\n",
    "    'base': (price_output - cost_output_base) / price_output,\n",
    "    'high': (price_output - cost_output_low) / price_output,\n",
    "}\n",
    "\n",
    "margin_input, margin_output"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "margin_table = pd.DataFrame([\n",
    "    ['Input', price_input, cost_input_low, cost_input_base, cost_input_high, margin_input['low'], margin_input['base'], margin_input['high']],\n",
    "    ['Output', price_output, cost_output_low, cost_output_base, cost_output_high, margin_output['low'], margin_output['base'], margin_output['high']],\n",
    "], columns=['Metric', 'Price', 'Cost Low', 'Cost Base', 'Cost High', 'Gross Margin Low', 'Gross Margin Base', 'Gross Margin High'])\n",
    "\n",
    "margin_table"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3) Sensitivity analysis\n",
    "\n",
    "We model a blended request mix and compute gross margin under different utilization assumptions.\n",
    "If output share is higher, margin compresses because output tokens are costlier to serve."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def blended_margin(input_share, output_share, cost_input, cost_output):\n",
    "    revenue = input_share * price_input + output_share * price_output\n",
    "    cost = input_share * cost_input + output_share * cost_output\n",
    "    return (revenue - cost) / revenue\n",
    "\n",
    "mixes = [\n",
    "    {'name': '80/20 input/output', 'input_share': 0.8, 'output_share': 0.2},\n",
    "    {'name': '50/50 input/output', 'input_share': 0.5, 'output_share': 0.5},\n",
    "    {'name': '20/80 input/output', 'input_share': 0.2, 'output_share': 0.8},\n",
    "]\n",
    "\n",
    "rows = []\n",
    "for mix in mixes:\n",
    "    for scenario, ci, co in [\n",
    "        ('low cost', cost_input_low, cost_output_low),\n",
    "        ('base cost', cost_input_base, cost_output_base),\n",
    "        ('high cost', cost_input_high, cost_output_high),\n",
    "    ]:\n",
    "        gm = blended_margin(mix['input_share'], mix['output_share'], ci, co)\n",
    "        rows.append({\n",
    "            'mix': mix['name'],\n",
    "            'scenario': scenario,\n",
    "            'gross_margin': gm\n",
    "        })\n",
    "\n",
    "sensitivity_df = pd.DataFrame(rows)\n",
    "sensitivity_df.pivot(index='mix', columns='scenario', values='gross_margin').round(3)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Simple chart\n",
    "fig, ax = plt.subplots(figsize=(8,4))\n",
    "plot_df = sensitivity_df.copy()\n",
    "plot_df['gross_margin_pct'] = plot_df['gross_margin'] * 100\n",
    "for scenario in ['low cost', 'base cost', 'high cost']:\n",
    "    subset = plot_df[plot_df['scenario'] == scenario]\n",
    "    ax.plot(subset['mix'], subset['gross_margin_pct'], marker='o', label=scenario)\n",
    "\n",
    "ax.set_ylabel('Gross margin, %')\n",
    "ax.set_xlabel('Token mix')\n",
    "ax.set_title('Atlas Cloud margin sensitivity on DeepSeek V4 Pro')\n",
    "ax.legend()\n",
    "plt.xticks(rotation=0)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},\n   "source": [
    "## 4) Interpretation helper\n",
    "\n",
    "We can estimate whether Atlas Cloud is likely profitable, breakeven, or loss-making under the base case.\n",
    "If base-case gross margin is positive across plausible mixes, the service is likely profitable."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},\n   "source": [
    "base_mix = {'input_share': 0.7, 'output_share': 0.3}\n",
    "base_gm = blended_margin(base_mix['input_share'], base_mix['output_share'], cost_input_base, cost_output_base)\n",
    "\n",
    "result = {\n",
    "    'base_mix': base_mix,\n",
    "    'base_gross_margin': base_gm,\n",
    "    'interpretation': 'profitable' if base_gm > 0 else 'breakeven_or_loss'\n",
    "}\n",
    "result"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},\n   "source": [
    "## 5) Takeaways\n",
    "\n",
    "- OpenAI: ~1.1M GPU-eq. base case, inference already roughly comparable to training.\n",
    "- Anthropic: ~350k GPU-eq. base case, inference share higher than OpenAI.\n",
    "- Atlas Cloud: public prices imply positive gross margin under reasonable serving costs."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
